# AB

In [2]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import json

In [3]:
# Omtoveren van xlsb naar csv 
data_excel = pd.read_excel("Data.xlsb")
data_reactie = pd.read_excel("Data_reacties.xlsb", sheet_name=1)

data_excel.to_csv("raw_data_incident.csv", index=False)
data_reactie.to_csv("raw_data_antwoorden.csv", index=False)

print(data_excel.head(5))
print(data_reactie.head(5))

   Unnamed: 0    Unnamed: 1 Unnamed: 2   Unnamed: 3 Unnamed: 4   Unnamed: 5  \
0  Incidenten           NaN        NaN          NaN        NaN          NaN   
1         NaN           NaN        NaN          NaN        NaN          NaN   
2        Type    Ingestuurd  Onderwerp  Toelichting    Kenmerk  Foutmelding   
3          10  45717.000995        NaN          NaN        NaN          NaN   
4          10  45717.001759        NaN          NaN        NaN          NaN   

  Unnamed: 6  
0        NaN  
1        NaN  
2     Status  
3        NaN  
4        NaN  
     Unnamed: 0                                         Unnamed: 1  \
0      Reacties                                                NaN   
1           NaN                                                NaN   
2    Ingestuurd                                            Reactie   
3  45717.000995  Goedemorgen André,\r\n\r\nHierbij jullie rappo...   
4  45717.001759  **Antwoord gegenereerd door Jonas:** ### SLA R...   

   Unnamed: 2 

In [4]:
# Columns veranderen van naam en irrelevante weghalen
# Incident
df = pd.read_csv("raw_data_incident.csv")
df = df.rename(columns={"Unnamed: 1": "Datum"})
df = df.rename(columns={"Unnamed: 0": "Type"})
df = df.rename(columns={"Unnamed: 2": "Onderwerp"})
df = df.rename(columns={"Unnamed: 3": "Toelichting"})
df = df.rename(columns={"Unnamed: 4": "Kenmerk"})
df = df.rename(columns={"Unnamed: 5": "Foutmelding"})
df = df.rename(columns={"Unnamed: 6": "Status"})

df = df.drop(columns=["Type"])
df = df.drop(columns=["Foutmelding"])
df = df.drop(index=[0, 1, 2])

print(df.isna().sum())
df.to_csv("raw_data_incident.csv", index=False)
print(df.head())

# Reactie
dff = pd.read_csv("raw_data_antwoorden.csv")
dff = dff.rename(columns={"Unnamed: 1": "Reactie"})
dff = dff.rename(columns={"Unnamed: 0": "Datum"})
dff = dff.rename(columns={"Unnamed: 2": "Labelcode"})
dff = dff.rename(columns={"Unnamed: 3": "Labeltekst"})
dff = dff.rename(columns={"Unnamed: 4": "Gebruiker"})

dff= dff.drop(index=[0, 1, 2])
dff = dff.drop(columns=["Labelcode"])
dff = dff.drop(columns=["Gebruiker"])


print(dff.isna().sum())
dff.to_csv("raw_data_antwoorden.csv", index=False)
print(dff.head())

Datum              0
Onderwerp      40000
Toelichting    40237
Kenmerk        40000
Status         40000
dtype: int64
                Datum Onderwerp Toelichting Kenmerk Status
3   45717.00099537037       NaN         NaN     NaN    NaN
4  45717.001759259256       NaN         NaN     NaN    NaN
5  45717.386145833334       NaN         NaN     NaN    NaN
6  45717.394212962965       NaN         NaN     NaN    NaN
7  45717.449270833335       NaN         NaN     NaN    NaN
Datum         0
Reactie       0
Labeltekst    0
dtype: int64
                Datum                                            Reactie  \
3   45717.00099537037  Goedemorgen André,\r\n\r\nHierbij jullie rappo...   
4  45717.001759259256  **Antwoord gegenereerd door Jonas:** ### SLA R...   
5  45717.001759259256  Goedemorgen André,\r\n\r\nHierbij jullie rappo...   
6   45719.36572916667  Beste Leroy,\r\n\r\nJe kunt de vrije tabel Per...   
7   45719.37894675926  Hoi Songul,\r\n\r\nBedankt voor je toelichting...   

          

In [5]:
# Mergen van de CSV bestanden gebaseerd op datum
df1 = pd.read_csv("raw_data_antwoorden.csv")
df2 = pd.read_csv("raw_data_incident.csv")
df_merged = pd.merge(df1, df2, on="Datum", how="inner")
df_merged.to_csv("Cleaned_data2.csv", index=False)

In [6]:
# Datum van excel format naar 01-01-2000
df = pd.read_csv("Cleaned_data2.csv")
df["Datum"] = pd.to_datetime(df["Datum"], unit="d", origin="1899-12-30")
df["Datum"] = pd.to_datetime(df["Datum"]).dt.strftime("%d-%m-%y")

df = df.dropna(subset=["Onderwerp"])
df = df[df["Status"] == "Afgehandeld"].reset_index(drop=True)

df.to_csv("Cleaned_data2.csv", index=False)
print(df.head(25))

       Datum                                            Reactie  \
0   07-04-25  Beste Marcel,\r\n\r\nOp dit moment hebben wij ...   
1   07-04-25  Beste Niels,\r\n\r\nWil jij de volgende stappe...   
2   07-04-25  Beste Gert,\r\n\r\nGoed elkaar gesproken te he...   
3   07-04-25  Beste Dennis,\r\n\r\nDe status UPA stond helem...   
4   07-04-25  Beste Isabella,\r\n\r\nEven een aannamen dat i...   
5   07-04-25  Beste Els,\r\n\r\nDit probleem wordt veroorzaa...   
6   07-04-25  **Antwoord gegenereerd door Jonas:** ### Forec...   
7   08-04-25  Goedemiddag Jan,\r\n\r\nZowaar, we troffen elk...   
8   08-04-25  Goedemorgen Petra,\r\n\r\nWe hebben geanalysee...   
9   08-04-25  Beste Chris,\r\n\r\nGoed elkaar even gesproken...   
10  08-04-25  Beste Ilse,\r\n\r\nIk heb meegekeken en heb de...   
11  08-04-25  Beste Wouter,\r\n\r\nZoals besproken tijdens h...   
12  08-04-25  **Antwoord gegenereerd door Jonas:** ### Hoe k...   
13  08-04-25  Dear Mirold,\r\n\r\nFirst in Dutch, for Englis..

In [7]:
df = pd.read_csv("Cleaned_data2.csv")
print(df["Kenmerk"].unique())

['HRM' 'Algemeen' 'Financieel' 'CRM' 'Projecten' 'Flex' 'Abonnementen'
 'Ordermanagement' 'Pocket' 'Small Business' 'Fiscaal' 'SB ondernemen'
 'België Franstalig']


In [8]:
# Kenmerken indelen in HRM, ERP, Algemeen
df = pd.read_csv("Cleaned_data2.csv")

df["Kenmerk"] = df["Kenmerk"].replace(["CRM", "Flex", "Pocket", "België Franstalig"],"Algemeen")
df["Kenmerk"] = df["Kenmerk"].replace(["Financieel", "Projecten", "Abonnementen", "Ordermanagement", "Small Business", "Fiscaal", "SB ondernemen"],"ERP")

df.to_csv("Cleaned_data2.csv", index=False)
print(df["Kenmerk"].unique())
print(df["Kenmerk"].value_counts())


['HRM' 'Algemeen' 'ERP']
Kenmerk
ERP         241
HRM         236
Algemeen    194
Name: count, dtype: int64


In [9]:
# Onnodige spaties verwijderen
df = pd.read_csv("Cleaned_data2.csv")
df["Toelichting"] = (
    df["Toelichting"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\*+", "", regex=True)
)

df["Reactie"] = (
    df["Reactie"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\*+", "", regex=True)
)
df.to_csv("Cleaned_data2.csv", index=False)

In [10]:
# Alleen de relevante reacties behouden (Niet de gegenereerde door Jonas)
df = pd.read_csv("Cleaned_data2.csv")
df = df[~df["Reactie"].str.startswith("Antwoord gegenereerd door Jonas:", na=False)]
df.to_csv("Cleaned_data2.csv", index=False)

In [11]:
df = pd.read_csv("Cleaned_data2.csv")
print(df["Status"].value_counts())
print(df["Labeltekst"].value_counts())

Status
Afgehandeld    625
Name: count, dtype: int64
Labeltekst
Goed antwoord Jonas    625
Name: count, dtype: int64


In [12]:
data = pd.read_csv("Cleaned_data2.csv")

data = data.drop(columns=["Labeltekst"])
data = data.drop(columns=["Datum"])
data = data.drop(columns=["Status"])
data = data.drop(columns=["Reactie"])
#data = data.drop(columns=["Toelichting"])

data.to_csv("Cleaned_data2.csv", index=False)

In [19]:
df = pd.read_csv("Cleaned_data2.csv")
print(df["Kenmerk"].unique())
print(df["Kenmerk"].value_counts())

['HRM' 'Algemeen' 'ERP']
Kenmerk
ERP         234
HRM         208
Algemeen    183
Name: count, dtype: int64


# Complete dataset maken
- ERP + HRM + Algemeen --> AB_Complete
- ERP + HRM --> AB_ERPandHRM

In [ ]:
import pandas as pd
#df = pd.read_csv("Cleaned_data2.csv")
df1 = pd.read_csv("GPT-5_notclean.csv")
df2 = pd.read_csv("GPT-o3_notclean.csv")
new_df = pd.merge(df1, df2, on=["Index"], how='inner')
new_df.to_csv("AB_complete.csv", index=False)

In [ ]:
# 5.1 
import pandas as pd
df2 = pd.read_csv("GPT5.1_low_notclean.csv")
df3 = pd.read_csv("GPT-o3_notclean.csv")
new_df = pd.merge(df2, df3, on=["Index"], how='inner')
new_df.to_csv("AB_complete_5.1.csv", index=False)

In [26]:
data = pd.read_csv("AB_complete.csv")
data2 = pd.read_csv("Cleaned_data2.csv")
final_data = pd.concat([data, data2], axis=1)
final_data.to_csv("AB_complete.csv", index=False)

In [4]:
# 5.1
data = pd.read_csv("AB_complete_5.1.csv")
data2 = pd.read_csv("Cleaned_data2.csv")
final_data = pd.concat([data, data2], axis=1)
final_data.to_csv("AB_complete_5.1.csv", index=False)

In [27]:
data = pd.read_csv("AB_complete.csv")
data = data[["Index", "Kenmerk", "Onderwerp", "Toelichting", "GPTo3_antwoorden","GPT5_antwoorden"]]
data.to_csv("AB_complete.csv", index=False)

In [7]:
# 5.1 
data = pd.read_csv("AB_complete_5.1.csv")
data = data[["Index", "Kenmerk", "Onderwerp", "Toelichting", "GPTo3_antwoorden","GPT-5.1_low_antwoorden"]]
data.to_csv("AB_complete_5.1.csv", index=False)

In [28]:
# Duplicaten verwijderen
data = pd.read_csv("AB_complete.csv")
data[data.duplicated(subset="Onderwerp", keep=False)]
no_duplicates = data.drop_duplicates(subset="Onderwerp", keep="first")

no_duplicates.to_csv("AB_complete.csv", index=False)

In [8]:
# Duplicaten verwijderen 5.1
data = pd.read_csv("AB_complete_5.1.csv")
data[data.duplicated(subset="Onderwerp", keep=False)]
no_duplicates = data.drop_duplicates(subset="Onderwerp", keep="first")

no_duplicates.to_csv("AB_complete_5.1.csv", index=False)

In [29]:
data = pd.read_csv("AB_complete.csv")
data[data.duplicated(subset="Onderwerp", keep=False)]

,Index,Kenmerk,Onderwerp,Toelichting,GPTo3_antwoorden,GPT5_antwoorden


In [30]:
df = pd.read_csv("AB_complete.csv")
print(df["Kenmerk"].unique())
print(df["Kenmerk"].value_counts())

['HRM' 'Algemeen' 'ERP']
Kenmerk
ERP         225
HRM         205
Algemeen    178
Name: count, dtype: int64


# AB_ERPandHRM

In [ ]:
# 5.1 
data = pd.read_csv("AB_complete_5.1.csv")
data = data[~data["Kenmerk"].str.startswith("Algemeen", na=False)]
data.to_csv("AB_ERPandHRM_5.1.csv", index=False)

print(data["Kenmerk"].value_counts())

Kenmerk
ERP    225
HRM    205
Name: count, dtype: int64


In [ ]:
data = pd.read_csv("AB_complete_5.1.csv")
data = data[~data["Kenmerk"].str.startswith("Algemeen", na=False)]
data.to_csv("AB_ERPandHRM.csv", index=False)

print(data["Kenmerk"].value_counts())

# ERP en HRM

In [10]:
# ERP dataset maken
data = pd.read_csv("AB_complete.csv")
data = data[~data["Kenmerk"].str.startswith("HRM", na=False)]
data = data[~data["Kenmerk"].str.startswith("Algemeen", na=False)]
data.to_csv("AB_ERP.csv", index=False)

print(data["Kenmerk"].value_counts())

Kenmerk
ERP    225
Name: count, dtype: int64


In [11]:
# Eventuele duplicaten verwijderen
data = pd.read_csv("AB_ERP.csv")
data[data.duplicated(subset="Onderwerp", keep=False)]
no_duplicates = data.drop_duplicates(subset="Onderwerp", keep="first")
no_duplicates.to_csv("AB_ERP.csv", index=False)
print(no_duplicates["Kenmerk"].value_counts())

Kenmerk
ERP    225
Name: count, dtype: int64


In [12]:
# HRM dataset maken
data = pd.read_csv("AB_complete.csv")
data = data[~data["Kenmerk"].str.startswith("ERP", na=False)]
data = data[~data["Kenmerk"].str.startswith("Algemeen", na=False)]
data.to_csv("AB_HRM.csv", index=False)

print(data["Kenmerk"].value_counts())

Kenmerk
HRM    205
Name: count, dtype: int64


In [13]:
data = pd.read_csv("AB_HRM.csv")
data[data.duplicated(subset="Onderwerp", keep=False)]
no_duplicates = data.drop_duplicates(subset="Onderwerp", keep="first")
no_duplicates.to_csv("AB_HRM.csv", index=False)

print(no_duplicates["Kenmerk"].value_counts())

Kenmerk
HRM    205
Name: count, dtype: int64
